In [ ]:
import os
import sys

if sys.platform.startswith("win"):
    openslide_path = os.getenv("OPENSLIDE_PATH")
    if openslide_path is None:
        raise EnvironmentError("Environment variable OPENSLIDE_PATH is not set")
    if not hasattr(os, "add_dll_directory"):
        raise RuntimeError("os.add_dll_directory is unavailable. Python >= 3.8 is required on Windows.")

    os.add_dll_directory(openslide_path)

import openslide

# 1. Visualize Slides Patches

## 1.1. Get Number of Tissue Patches

In [ ]:
from wsi_manager.tissue import SlideTissueProcessor

slide_path = '/path/to/slide.svs'
region_x, region_y = 7000, 21000
region_width, region_height = 20000, 20000

processor = SlideTissueProcessor(slide_path,
                                 x=region_x, y=region_y,
                                 region_width=region_width, region_height=region_height,
                                 tissue_method='gradient')

processor.process()

## 1.2. Plot Slide with Patches Detected

In [ ]:
processor.plot_tissue_regions(tissuecolor='blue',
                            tissuealpha=0.1,
                            edgecolor='black',
                            edgewidth=1)

# 2. Crop Analysis

In [ ]:
import platform
import json
import ast
import pandas as pd
import numpy as np
from wsi_manager.crop import CropAnalysis

OS_MODE = platform.system()

try:
    with open("_info/config.json", 'r') as file:
        conf = json.load(file)
except:
    raise FileNotFoundError("Couldn't Load Config File. Check if the `config.json` file exists under `_info` directory.")

with open(os.path.join(conf["info_dir"], "tree_pair_dict.json"), "r") as file:
    tree_pair_dict = json.load(file)

In [ ]:
eval_id = 'eval_id'
stage_eval = 'Root' # 'Node', 'Leaf1', 'Leaf2'

metrics_save_dir = os.path.join(
    conf["results_dir"],
    "crop_eval",
    eval_id,
    "metrics_plot"
)

crop_pred_df = pd.read_csv(
    os.path.join(
        conf["results_dir"],
        "crop_eval",
        eval_id,
        f"{stage_eval}.csv"
    )
)

crop_df = crop_pred_df[['type', 'label']].copy()
crop_df['pred_probs'] = crop_pred_df['preds'].apply(ast.literal_eval)

### Uncertainty Analysis on Crops

In [ ]:
crop_evaluator = CropAnalysis(stage=stage_eval)
crop_df = crop_evaluator.find_pred_labels(input=crop_df)
crop_df = crop_evaluator.get_correct(input=crop_df)
crop_df = crop_evaluator.apply_metrics(input=crop_df, alpha=0.1, bins=50)

In [ ]:
# Plot Different Metrics
crop_evaluator.plot_metrics(input=crop_df,
                            savefig=True,
                            save_dir=metrics_save_dir,
                            figsize=(10, 20),
                            fig_alpha=0.5,
                            bins=50,
                            density=True)

In [ ]:
# Plot Renyi Entropy Distances for Different Alpha Values
crop_evaluator.plot_dist_vs_renyi_alpha(input=crop_df,
                                    alpha_range=np.linspace(0.05, 0.95, 100),
                                    savefig=True,
                                    save_dir=metrics_save_dir,
                                    bins=50,
                                    figsize=(14, 4),
                                    marker='o')

In [ ]:
# Plot Renyi Entropy Distances for Different Alpha Values
crop_evaluator.plot_params_vs_renyi_ents(input=crop_df,
                                    ent_range=np.linspace(8.16, 8.27, 100),
                                    savefig=True,
                                    save_dir=metrics_save_dir,
                                    bins=50,
                                    figsize=(15, 10),
                                    marker='o')

In [ ]:
crop_evaluator.apply_uncertainty(crop_df, 
                                 threshold=8.2389, 
                                 metric='renyi_ent', 
                                 alpha=crop_evaluator.max_alpha,
                                 bins=50)

### Write Parameters Based on Above Evaluations

#### Copy this into the '_info/model_params.json' for Considered Stage and Evaluation Id

In [ ]:
params = {
                "iter":10,
                "metric":"renyi_entropy",
                "alpha":0.9500,
                "norm":true,
                "num_bins":50,
                "threshold":8.2389,
                "inequality":"U<=T"
}